# SSE Example with htmx v4

In [1]:
import random
from asyncio import sleep
from fasthtml.common import *
from fasthtml.jupyter import *

app,rt = fast_app(htmx=False, htmx4=True, exts='sse')

@rt
def index():
    return Titled("SSE Random Number Generator",
        P("Generate random numbers, as the list grows scroll downwards."),
        Div(hx_get="/number-stream",
            hx_trigger="load",
            hx_swap="beforeend show:bottom"))

shutdown_event = signal_shutdown()

async def number_generator():
    while not shutdown_event.is_set():
        data = Article(random.randint(1, 100))
        yield sse_message(data, htmx4=True)
        await sleep(1)

@rt("/number-stream")
async def get(): return EventStream(number_generator())

srv = JupyUvi(app)